In [4]:
import torch
from IPython  import display
from d2l import torch as d2l
batch_size=256
train_iter,test_iter=d2l.load_data_fashion_mnist(batch_size)

In [5]:
num_input=784
num_output=10
w=torch.normal(0,0.01,size=(num_input,num_output),requires_grad=True)
b=torch.zeros(num_output,requires_grad=True)

In [6]:
def softmax(X):
    X_exp = torch.exp(X)
    partition = X_exp.sum(1, keepdim=True)
    return X_exp / partition  # 这里应用了广播机制

In [7]:
def net(X):
    return softmax(torch.matmul(X.reshape((-1, w.shape[0])), w) + b)#将图片展平

In [8]:
def cross_entropy(y_hat, y):#交叉熵损失函数
    return -torch.log(y_hat[range(len(y_hat)), y])

In [9]:
def accuracy(y_hat, y):
    """计算预测正确的数量"""
    if len(y_hat.shape) > 1 and y_hat.shape[1] > 1:
        y_hat = y_hat.argmax(axis=1)
    cmp = (y_hat.type(y.dtype) == y)
    return float(cmp.type(y.dtype).sum())

In [10]:
class Accumulator:
    """在n个变量上累加"""
    def __init__(self, n):
        self.data = [0.0] * n
        
    def add(self, *args):
        self.data = [a + float(b) for a, b in zip(self.data, args)]
        
    def reset(self):
        self.data = [0.0] * len(self.data)
        
    def __getitem__(self, idx):
        return self.data[idx]

In [11]:
def train_epoch_ch3(net, train_iter, loss, updater):
    """训练模型一个迭代周期"""
    # 如果是 PyTorch 的 Module，设置为训练模式
    if isinstance(net, torch.nn.Module):
        net.train()
    
    # 累加器：损失总和，正确预测数，样本总数
    metric = Accumulator(3)
    
    for X, y in train_iter:
        # 前向传播
        y_hat = net(X)
        l = loss(y_hat, y)
        
        # 判断 updater 类型
        if isinstance(updater, torch.optim.Optimizer):
            # 使用 PyTorch 优化器
            updater.zero_grad()   # 梯度清零
            l.backward()          # 反向传播
            updater.step()        # 更新参数
            metric.add(float(l) * len(y), accuracy(y_hat, y), y.numel())
        else:
            # 使用自定义优化器
            l.sum().backward()
            updater(X.shape[0])
            metric.add(float(l.sum()), accuracy(y_hat, y), y.numel())
    
    # 返回：平均损失，准确率
    return metric[0] / metric[2], metric[1] / metric[2]

In [12]:
class Animator:
    """在动画中绘制数据"""
    def __init__(self, xlabel=None, ylabel=None, legend=None, xlim=None,
                 ylim=None, xscale='linear', yscale='linear',
                 fmts=('-', 'm--', 'g-.', 'r:'), nrows=1, ncols=1,
                 figsize=(3.5, 2.5)):
        # 设置图例
        if legend is None:
            legend = []
        
        # 使用 SVG 格式显示图形
        d2l.use_svg_display()
        
        # 创建子图
        self.fig, self.axes = d2l.plt.subplots(nrows, ncols, figsize=figsize)
        
        # 如果是单图，将 axes 转换为列表
        if nrows * ncols == 1:
            self.axes = [self.axes]
        
        # 配置坐标轴
        self.config_axes = lambda: d2l.set_axes(
            self.axes[0], xlabel, ylabel, xlim, ylim, xscale, yscale, legend
        )
        
        # 初始化数据存储
        self.X, self.Y, self.fmts = None, None, fmts
    
    def add(self, x, y):
        """添加数据点"""
        # 如果 y 不是列表，转换为列表
        if not hasattr(y, "__len__"):
            y = [y]
        
        n = len(y)
        # 初始化 X, Y
        if not self.X:
            self.X = [[] for _ in range(n)]
            self.Y = [[] for _ in range(n)]
        
        # 添加数据
        for i, (a, b) in enumerate(zip(self.X, self.Y)):
            a.append(x)
            b.append(y[i])
        
        # 清空并重绘
        self.axes[0].cla()
        
        # 绘制所有曲线
        for x_vals, y_vals, fmt in zip(self.X, self.Y, self.fmts):
            self.axes[0].plot(x_vals, y_vals, fmt)
        
        # 重新配置坐标轴
        self.config_axes()
        
        # 显示图形
        d2l.plt.draw()
        d2l.plt.pause(0.001)

In [13]:
def evaluate_accuracy(net, data_iter):
    """计算在指定数据集上模型的精度"""
    # 如果是 PyTorch 的 Module，设置为评估模式
    if isinstance(net, torch.nn.Module):
        net.eval()  # 关闭 dropout 和 batch norm 的训练行为
    
    # 累加器：正确预测数，预测总数
    metric = Accumulator(2)
    
    # 遍历数据
    for X, y in data_iter:
        metric.add(accuracy(net(X), y), y.numel())
    
    # 返回准确率 = 正确预测数 / 总样本数
    return metric[0] / metric[1]

In [14]:
def train_ch3(net, train_iter, test_iter, loss, num_epochs, updater):
    """训练模型"""
    # 创建动画可视化器
    animator = Animator(xlabel='epoch', xlim=[1, num_epochs], ylim=[0.3, 0.9],
                        legend=['train loss', 'train acc', 'test acc'])
    
    for epoch in range(num_epochs):
        # 训练一个 epoch，返回训练损失和训练准确率
        train_metrics = train_epoch_ch3(net, train_iter, loss, updater)
        
        # 在测试集上评估准确率
        test_acc = evaluate_accuracy(net, test_iter)
        
        # 添加数据到动画中
        animator.add(epoch + 1, train_metrics + (test_acc,))
    
    # 解包训练指标
    train_loss, train_acc = train_metrics
    
    # 断言训练结果合理
    assert train_loss < 0.5, f"训练损失过高: {train_loss}"
    assert train_acc > 0.7, f"训练准确率过低: {train_acc}"
    assert test_acc > 0.7, f"测试准确率过低: {test_acc}"

In [2]:
lr = 0.1

def updater(batch_size):
    return d2l.sgd([w, b], lr, batch_size)

In [3]:
num_epochs = 10
train_ch3(net, train_iter, test_iter, cross_entropy, num_epochs,updater)

NameError: name 'train_ch3' is not defined